In [12]:
import statsmodels.api as sm
import numpy as np
from scipy import stats
import pandas as pd

## Question 1

Which is closest to the probability that this t-test will be able to detect the event at event_time + 1 with the given code? 

Option A
0.7

Option B
0.3

Option C
0.1

Option D
0.5

In [2]:
num = 1000

event_time = int(num / 2)

R_market = np.random.normal(0, 1, num) + np.arange(num) / num
 
R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2
 
results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()
 
alpha, beta = results.params 
 
resid = R_target - results.predict(sm.add_constant(R_market)) 
 
print(resid[event_time + 1] / resid[:event_time].std(ddof = 2)) 

3.4561664749440864


In [3]:
t_stats = []
for _ in range(1_000):
    num = 1000

    event_time = int(num / 2)

    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2
    
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()
    
    alpha, beta = results.params 
    
    resid = R_target - results.predict(sm.add_constant(R_market)) 
    
    # print(resid[event_time + 1] / resid[:event_time].std(ddof = 2)) 
    t_stats.append(resid[event_time + 1] / resid[:event_time].std(ddof = 2))

In [4]:
stats.norm.ppf(0.975) # for a standard 0.05 significance level subtract 0.025 from each side

np.float64(1.959963984540054)

In [5]:
print("Mean t-stat:", np.mean(t_stats))
print("Std t-stat:", np.std(t_stats))
power_05 = np.mean(np.abs(t_stats) > 1.96)
print("Power at 5%:", power_05)

Mean t-stat: 2.029583781257399
Std t-stat: 1.0393676808799532
Power at 5%: 0.519


D. 0.5

## Question 2

Use the same code but put np.random.seed(0) at the beginning of each loop to ensure that you are performing placebo tests on a fixed dataset. Perform a placebo test by setting the fictitious event_time to all possible times, while leaving the event in R_target at just the 1 time. The placebo test trains itself on the data leading up to the fictitious event. About what fraction of placebo tests seem to detect an event at the fictitious event time?

Option A
0.05

Option B
0.02

Option C
0.10

Option D
0.15

In [6]:
num = 1000
event_time_true = int(num / 2)

np.random.seed(0)
R_market = np.random.normal(0, 1, num) + np.arange(num) / num
R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == event_time_true + 1) * 2

significant_count = 0
valid_tests = 0

# Test all possible fictitious event times. 
# The placebo test trains itself on data leading up to the fictitious event time (indices 0 to t-1).
# To run sm.OLS, we need at least a few data points, say t >= 3.
# It tests for an event at the fictitious event_time + 1. So event_time + 1 < num => event_time <= 998.
for event_time in range(3, num - 1):
    # Train the model on data leading up to the fictitious event time
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()

    # Extract parameters and residuals
    alpha, beta = results.params 
    resid = R_target - results.predict(sm.add_constant(R_market)) 

    # Calculate t-statistic at event_time + 1
    # std is calculated over the estimation window :event_time
    # ddof = 2 matching the original code
    std_estimation = resid[:event_time].std(ddof=2)
    
    t_stat = resid[event_time + 1] / std_estimation
    if abs(t_stat) > 1.96:
        significant_count += 1
    valid_tests += 1
print(f"Significant: {significant_count}, Total valid tests: {valid_tests}, Fraction: {significant_count / valid_tests}")

Significant: 47, Total valid tests: 996, Fraction: 0.04718875502008032


A. 0.05

## Question 3

Do the same placebo test, but this time only run the test 20 times before and twenty times after the actual event. On average (over many runs of the code), what fraction of the 40 placebo tests get a higher t-value than the actual event? This time, adjust np.random.seed() to represent a different dataset when needed. 

Option A
0.15

Option B
0.25

Option C
0.35

Option D
0.05

In [7]:
def run_placebo_test(seed):
    np.random.seed(seed)
    num = 1000
    event_time_actual = int(num / 2) # 500

    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == event_time_actual + 1) * 2
    
    # Actual event t-value at event_time + 1
    results_actual = sm.OLS(R_target[:event_time_actual], sm.add_constant(R_market[:event_time_actual])).fit()
    resid_actual = R_target - results_actual.predict(sm.add_constant(R_market))

    std_estimation_actual = resid_actual[:event_time_actual].std(ddof=2)
    t_stat_actual = resid_actual[event_time_actual + 1] / std_estimation_actual
    
    # run the test 20 times before and twenty times after the actual event
    fict_times = list(range(event_time_actual - 20, event_time_actual)) + list(range(event_time_actual + 1, event_time_actual + 21))
    
    higher_count = 0
    for f_time in fict_times:
        results_fict = sm.OLS(R_target[:f_time], sm.add_constant(R_market[:f_time])).fit()
        resid_fict = R_target - results_fict.predict(sm.add_constant(R_market))
        std_estimation_fict = resid_fict[:f_time].std(ddof=2)
        
        t_stat_fict = abs(resid_fict[f_time + 1] / std_estimation_fict)
        
        if t_stat_fict > t_stat_actual:
            higher_count += 1
            
    return higher_count / len(fict_times)

fractions = [run_placebo_test(seed) for seed in range(1_000)]
print(np.mean(fractions))


0.150325


A. 0.15

## Question 4

Do the same thing as in question 2, but this time use make_error with corr_const = 0.9 to generate the error for R_target instead of np.random.normal. Consider before attempting this: Would you expect this kind of dataset, where errors are not independent, to result in more or fewer false positives in the placebo tests? 

Option A
0.45

Option B
0.25

Option C
0.05

Option D
0.65

In [8]:
def make_error(corr_const, num): 
  sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2)) 
  err = list() 
  prev = np.random.normal(0, sigma) 
  for n in range(num): 
    prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma) 
    err.append(prev) 
  return np.array(err) 

In [9]:
num = 1000
event_time_true = int(num / 2)

np.random.seed(0)
# R_market = np.random.normal(0, 1, num) + np.arange(num) / num
# R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == event_time_true + 1) * 2
# use make_error with corr_const = 0.9 to generate the error for R_target instead of np.random.normal
R_market = np.random.normal(0, 1, num) + np.arange(num) / num
error_term = make_error(0.9, num)
R_target = 2 + R_market + error_term + (np.arange(num) == event_time_true + 1) * 2

significant_count = 0
valid_tests = 0

# Test all possible fictitious event times. 
# The placebo test trains itself on data leading up to the fictitious event time (indices 0 to t-1).
# To run sm.OLS, we need at least a few data points, say t >= 3.
# It tests for an event at the fictitious event_time + 1. So event_time + 1 < num => event_time <= 998.
for event_time in range(3, num - 1):
    # Train the model on data leading up to the fictitious event time
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()

    # Extract parameters and residuals
    alpha, beta = results.params 
    resid = R_target - results.predict(sm.add_constant(R_market)) 

    # Calculate t-statistic at event_time + 1
    # std is calculated over the estimation window :event_time
    # ddof = 2 matching the original code
    std_estimation = resid[:event_time].std(ddof=2)
    
    t_stat = resid[event_time + 1] / std_estimation
    if abs(t_stat) > 1.96:
        significant_count += 1
    valid_tests += 1
print(f"Significant: {significant_count}, Total valid tests: {valid_tests}, Fraction: {significant_count / valid_tests}")


Significant: 47, Total valid tests: 996, Fraction: 0.04718875502008032


C. 0.05

## Reflection Question 1

Construct a dataset for an event study where the value, derivative, and second derivative of a trend all change discontinuously (suddenly) after an event.

Build a model that tries to decide whether the event is real (has a nonzero effect) using:

(a) only the value,

(b) the value, derivative, and second derivative.

Which of these models is better at detecting and/or quantifying the impact of the event?  (What might "better" mean here?)

In [15]:

# Setup a span of points from -10 to 10. This will act like our timeline 
n_points = 200
t = np.linspace(-10, 10, n_points)
# Event happens at 0.0 on the timeline
event_time = 0.0

# Base params (Before event)
beta_0, beta_1, beta_2 = 1, 2, 3  # Value, derivative, 2nd derivative / 2

# Introduce discontinuous changes to params
delta_0, delta_1, delta_2 = 2, 4, 6

# Build mask that says whether the event has already occurred at the index of the point
post = (t >= event_time).astype(int)

# We setup the equation y_true = beta_0 + beta_1 * t + beta_2 * t^2
# This shows the intercept (beta_0), the first derivative (beta_1), and the second derivative (beta_2)
# Before the event, post will cancel out all terms aside from beta_0, beta_1, and beta_2
# After the event, post will multiply the structural breaks (delta_0, delta_1, delta_2)
#  which gives us the discontinuous changes in the function.
y_true = (beta_0 + delta_0 * post) + \
         (beta_1 + delta_1 * post) * t + \
         (beta_2 + delta_2 * post) * (t**2)

# Add noise
y = y_true + np.random.normal(0, 1.0, size=n_points)

# Construct Dataset
df = pd.DataFrame({'t': t, 't2': t**2, 'post': post, 'y': y})
df['t_post'] = df['t'] * df['post']
df['t2_post'] = df['t2'] * df['post']

# --- Model A: Only allows value (intercept) to change ---
X_A = sm.add_constant(df[['t', 't2', 'post']])
model_A = sm.OLS(df['y'], X_A).fit()

# --- Model B: Allows value, derivative, and 2nd derivative to change ---
X_B = sm.add_constant(df[['t', 't2', 'post', 't_post', 't2_post']])
model_B = sm.OLS(df['y'], X_B).fit()

# --- Print Diagnostics ---
print(f"Model A (Intercept Only Shift) -> AIC: {model_A.aic:.2f} | Adj R2: {model_A.rsquared_adj:.4f}")
print(f"Model B (Full Derivative Shift) -> AIC: {model_B.aic:.2f} | Adj R2: {model_B.rsquared_adj:.4f}\n")

print(f"Model A - Event p-value (t-test): {model_A.pvalues['post']:.5f}")
# For Model B, we perform a joint F-test that all structural shift coefficients are zero
f_test = model_B.f_test("post = t_post = t2_post = 0")
print(f"Model B - Event p-value (Joint F-test): {f_test.pvalue:.5e}")


Model A (Intercept Only Shift) -> AIC: 1821.06 | Adj R2: 0.9914
Model B (Full Derivative Shift) -> AIC: 553.21 | Adj R2: 1.0000

Model A - Event p-value (t-test): 0.00000
Model B - Event p-value (Joint F-test): 5.83798e-301


## Reflection Question 2

Construct a dataset in which there are three groups whose values each increase discontinuously (suddenly) by the same amount at a shared event; they change in parallel
over time, but they have different starting values.  Create a model that combines group fixed effects with an event study, as suggested in the online reading.

Explain what you did, how the model works, and how it accounts for both baseline differences and the common event effect.
